In [5]:
# =================================================================================================
# Raw data : NHANES 2003-2014 Demographics, Laboratory, and Questionnaire datasets
# Purpose : Create one unified participant-level NHANES dataset by aggregating all domains using SEQN.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique ID used for merge.
# Files Used :
#   neighbourhood_nhanes_demographics_2003_2014_final.csv
#   neighbourhood_nhanes_labs_2003_2014_appended.csv
#   neighbourhood_nhanes_questionnaire_2003_2014_appended.csv
# Data Dictionary : NHANES 2003-2014 documentation
# Data Availability Matrix : Generated in this notebook using domain presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/
# Developer Notes : Full outer join is used so no SEQN is dropped from any file.
# File Creator : Neha Khairnar
# =================================================================================================

# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Define Paths
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data" / "processed_v3" / "2005_2006"
demo_file = DATA_DIR / "neighbourhood_nhanes_demographics_2005_2006_raw_corrected.csv"
labs_file = DATA_DIR / "neighbourhood_nhanes_labs_2005_2006_raw_corrected.csv"
questionnaire_file = DATA_DIR / "neighbourhood_nhanes_questionnaire_2005_2006_raw_corrected.csv"
Examination_file = DATA_DIR / "neighbourhood_nhanes_examination_2005_2006_raw_corrected.csv"
Dietary_file = DATA_DIR / "neighbourhood_nhanes_dietary_2005_2006_raw_corrected.csv"


# =================================================================================================
# 📥 Step 3: Load Files
# =================================================================================================

demo = pd.read_csv(demo_file, dtype=str, keep_default_na=False)
labs = pd.read_csv(labs_file, dtype=str, keep_default_na=False)
questionnaire = pd.read_csv(questionnaire_file, dtype=str, keep_default_na=False)
Examination = pd.read_csv(Examination_file, dtype=str, keep_default_na=False)
Dietary = pd.read_csv(Dietary_file, dtype=str, keep_default_na=False)

print("📊 Loaded Files:")
print("Demographics:", demo.shape)
print("Labs:", labs.shape)
print("Questionnaire:", questionnaire.shape)
print("Examination:", Examination.shape)
print("Dietary:", Dietary.shape)

# =================================================================================================
# 🧹 Step 4: Standardize Column Names
# =================================================================================================
demo.columns = demo.columns.str.upper()
labs.columns = labs.columns.str.upper()
questionnaire.columns = questionnaire.columns.str.upper()
Examination.columns = Examination.columns.str.upper()
Dietary.columns = Dietary.columns.str.upper()

# =================================================================================================
# ✅ Step 5: Validate SEQN Exists
# =================================================================================================

for name, df_check in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    if "SEQN" not in df_check.columns:
        raise ValueError(f"❌ SEQN column missing in {name} file")
    else:
        print(f"✅ SEQN found in {name}")

# =================================================================================================
# 📊 Step 6: File Structure Summary
# =================================================================================================
summary = []
for name, df_temp in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "unique_seqn": df_temp["SEQN"].nunique(),
        "duplicate_seqn_count": df_temp.duplicated(subset=["SEQN"]).sum(),
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)
display(summary_df[["file", "rows", "columns", "unique_seqn", "duplicate_seqn_count"]])

final_df = (
    demo
    .merge(questionnaire, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_QUESTIONNAIRE"))
    .merge(labs, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_LAB"))
    .merge(Examination, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_EXAMINATION"))
    .merge(Dietary, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_DIETARY"))
)
# =================================================================================================
# 👀 Step 10: Validate Final Unified Dataset
# =================================================================================================

print("📊 Final Unified Dataset Shape:", final_df.shape)
print("Unique SEQN Count:", final_df["SEQN"].nunique())
print("Duplicate SEQN Count:", final_df.duplicated(subset=["SEQN"]).sum())
display(final_df.head(20))

OUTPUT_DIR = BASE_DIR / "data" / "processed_v3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_unified_2005_2006_corrected.csv"
final_df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Final unified NHANES dataset saved to:")
print(OUTPUT_PATH)


📊 Loaded Files:
Demographics: (10348, 45)
Labs: (9440, 37)
Questionnaire: (10348, 468)
Examination: (9950, 27)
Dietary: (9950, 236)
✅ SEQN found in demographics
✅ SEQN found in labs
✅ SEQN found in questionnaire
✅ SEQN found in Examination
✅ SEQN found in Dietary


,file,rows,columns,unique_seqn,duplicate_seqn_count
0,demographics,10348,45,10348,0
1,labs,9440,37,9440,0
2,questionnaire,10348,468,10348,0
3,Examination,9950,27,9950,0
4,Dietary,9950,236,9950,0


📊 Final Unified Dataset Shape: (10348, 809)
Unique SEQN Count: 10348
Duplicate SEQN Count: 0


,SEQN,SDDSRVYR,RIDSTATR,RIDEXMON,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDAGEEX,RIDRETH1,DMQMILIT,...,DR2TP184,DR2TP204,DR2TP205,DR2TP225,DR2TP226,DR2_300,DR2_320Z,DR2_330Z,DR2BWATZ,DR2TWS
0,31127,4,2,2,1,0,11,12,3,,...,0.069,0.019,0.0,0.0,0.0,2,0.0,0.0,0.0,4
1,31128,4,2,1,2,11,132,132,4,,...,0.02,0.105,0.0,0.0,0.0,3,474.0,474.0,0.0,99
2,31129,4,2,2,1,15,189,190,4,,...,0.109,0.057999999999999996,0.088,0.016,0.077,2,710.4,0.0,710.4,1
3,31130,4,2,2,2,85,,,3,2,...,,,,,,,,,,
4,31131,4,2,2,2,44,535,536,4,2,...,0.0,0.108,0.002,0.001,0.019,1,903.56,903.56,0.0,1
5,31132,4,2,2,1,70,842,843,3,1,...,0.001,0.056,0.005,0.0,0.003,2,414.75,0.0,414.75,1
6,31133,4,2,2,2,16,193,194,4,,...,0.11,0.167,0.0,0.061,0.054,3,311.06,0.0,311.06,4
7,31134,4,2,2,1,73,882,883,3,1,...,0.0,0.172,0.009000000000000001,0.0,0.0,2,518.44,518.44,0.0,2
8,31135,4,2,2,1,0,10,11,5,,...,0.0,0.255,0.007,0.004,0.10699999999999998,2,148.13,148.13,0.0,1
9,31136,4,1,,2,41,493,,4,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



✅ Final unified NHANES dataset saved to:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\processed_v3\neighbourhood_nhanes_unified_2005_2006_corrected.csv
